# 1. Thu thập Dữ liệu (Data Ingestion)

Notebook này trình bày quá trình thu thập và kiểm tra chất lượng 9 file CSV từ bộ dữ liệu **Olist Brazilian E-Commerce**, sau đó đưa lên HDFS theo kiến trúc Medallion (Bronze Layer).

Trước khi phân tích, dữ liệu thô cần được kiểm tra tính toàn vẹn (schema validation) và chất lượng (null, trùng lặp) để đảm bảo không có lỗi lan truyền sang các bước sau.


## 1.1 Giới thiệu bộ dữ liệu Olist

Bộ dữ liệu Olist Brazilian E-Commerce chứa thông tin về **~100.000 đơn hàng** từ năm 2016-2018 trên sàn thương mại điện tử Olist (Brazil). Dữ liệu gồm **9 file CSV** liên kết với nhau:

| STT | File | Mô tả | Vai trò |
|-----|------|--------|---------|
| 1 | olist_orders_dataset.csv | Thông tin đơn hàng | Bảng chính (fact table) |
| 2 | olist_order_items_dataset.csv | Chi tiết sản phẩm trong đơn | Bảng chi tiết |
| 3 | olist_order_payments_dataset.csv | Thanh toán | Bảng phụ |
| 4 | olist_order_reviews_dataset.csv | Đánh giá | Bảng phụ |
| 5 | olist_customers_dataset.csv | Khách hàng | Dimension table |
| 6 | olist_products_dataset.csv | Sản phẩm | Dimension table |
| 7 | olist_sellers_dataset.csv | Người bán | Dimension table |
| 8 | olist_geolocation_dataset.csv | Vị trí địa lý | Dimension table |
| 9 | product_category_name_translation.csv | Dịch tên danh mục | Lookup table |

## 1.2 Kiến trúc Medallion (Bronze / Silver / Gold)

Chúng ta sử dụng kiến trúc **Medallion** (hay còn gọi là Data Lakehouse Architecture) gồm 3 tầng:

- **Bronze (Tầng đồng)**: Dữ liệu thô, chưa qua xử lý. Lưu nguyên bản CSV để có thể truy vết lại.
- **Silver (Tầng bạc)**: Dữ liệu đã được làm sạch, join các bảng, xử lý null/trùng lặp.
- **Gold (Tầng vàng)**: Dữ liệu đã được tổng hợp, sẵn sàng cho phân tích và ML.

> Ở bước Ingestion này, ta chỉ đưa dữ liệu vào **tầng Bronze**.

## 1.3 Import thư viện và cấu hình

In [1]:
import os
import csv
import json
import pandas as pd
from datetime import datetime

# Đường dẫn tới thư mục chứa dữ liệu CSV
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
# Nếu chạy trực tiếp trong notebooks/, DATA_DIR sẽ trỏ đến ../data/

# Đường dẫn HDFS Bronze (minh họa - cần Hadoop cluster để chạy thực tế)
HDFS_BRONZE = "/user/bigdata/olist/bronze"

print(f"Thư mục dữ liệu: {DATA_DIR}")
print(f"HDFS Bronze path: {HDFS_BRONZE}")

Thư mục dữ liệu: c:\Users\Admin\Desktop\olist-bigdata-project-nhom18\data
HDFS Bronze path: /user/bigdata/olist/bronze


## 1.4 Định nghĩa Schema cho từng file

Mỗi file CSV có một **schema** riêng, bao gồm:
- **Các cột bắt buộc** (required columns): phải có mặt trong file
- **Khóa chính** (primary key): phải là duy nhất, không trùng lặp
- **Thư mục HDFS đích**: nơi file sẽ được upload lên

### Tại sao cần schema validation?
- Đảm bảo file CSV không bị thiếu cột quan trọng
- Phát hiện sớm dữ liệu hỏng trước khi đưa vào pipeline

In [2]:
# Định nghĩa schema cho 9 file CSV
SOURCE_SCHEMA = {
    "olist_orders_dataset.csv": {
        "hdfs_folder": "orders",
        "required_columns": ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
        "primary_key": "order_id",
    },
    "olist_order_items_dataset.csv": {
        "hdfs_folder": "order_items",
        "required_columns": ["order_id", "product_id", "seller_id", "price", "freight_value"],
        "primary_key": None,  # Khóa tổng hợp (composite key)
    },
    "olist_order_payments_dataset.csv": {
        "hdfs_folder": "order_payments",
        "required_columns": ["order_id", "payment_type", "payment_value"],
        "primary_key": None,
    },
    "olist_order_reviews_dataset.csv": {
        "hdfs_folder": "order_reviews",
        "required_columns": ["order_id", "review_score"],
        "primary_key": "review_id",
    },
    "olist_customers_dataset.csv": {
        "hdfs_folder": "customers",
        "required_columns": ["customer_id", "customer_unique_id", "customer_city", "customer_state"],
        "primary_key": "customer_id",
    },
    "olist_products_dataset.csv": {
        "hdfs_folder": "products",
        "required_columns": ["product_id", "product_category_name"],
        "primary_key": "product_id",
    },
    "olist_sellers_dataset.csv": {
        "hdfs_folder": "sellers",
        "required_columns": ["seller_id", "seller_city", "seller_state"],
        "primary_key": "seller_id",
    },
    "olist_geolocation_dataset.csv": {
        "hdfs_folder": "geolocation",
        "required_columns": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
        "primary_key": None,
    },
    "product_category_name_translation.csv": {
        "hdfs_folder": "category_translation",
        "required_columns": ["product_category_name", "product_category_name_english"],
        "primary_key": "product_category_name",
    },
}

print(f"Tổng số file cần xử lý: {len(SOURCE_SCHEMA)}")
for fname, schema in SOURCE_SCHEMA.items():
    print(f" {fname} -> HDFS: {HDFS_BRONZE}/{schema['hdfs_folder']}/")

Tổng số file cần xử lý: 9
 olist_orders_dataset.csv -> HDFS: /user/bigdata/olist/bronze/orders/
 olist_order_items_dataset.csv -> HDFS: /user/bigdata/olist/bronze/order_items/
 olist_order_payments_dataset.csv -> HDFS: /user/bigdata/olist/bronze/order_payments/
 olist_order_reviews_dataset.csv -> HDFS: /user/bigdata/olist/bronze/order_reviews/
 olist_customers_dataset.csv -> HDFS: /user/bigdata/olist/bronze/customers/
 olist_products_dataset.csv -> HDFS: /user/bigdata/olist/bronze/products/
 olist_sellers_dataset.csv -> HDFS: /user/bigdata/olist/bronze/sellers/
 olist_geolocation_dataset.csv -> HDFS: /user/bigdata/olist/bronze/geolocation/
 product_category_name_translation.csv -> HDFS: /user/bigdata/olist/bronze/category_translation/


## 1.5 Kiểm tra sự tồn tại của các file

Trước khi xử lý, cần đảm bảo tất cả 9 file CSV đều có mặt trong thư mục `data/`.

In [3]:
print("=" * 60)
print("KIỂM TRA CÁC FILE CSV CÓ TRONG THƯ MỤC DATA KO")
print("=" * 60)

all_exist = True
for filename in SOURCE_SCHEMA.keys():
    filepath = os.path.join(DATA_DIR, filename)
    exists = os.path.exists(filepath)
    if exists:
        size_mb = round(os.path.getsize(filepath) / (1024 * 1024), 2)
        print(f" {filename} ({size_mb} MB)")
    else:
        print(f" {filename} - KHÔNG TÌM THẤY!")
        all_exist = False

print()
if all_exist:
    print("Tất cả 9/9 file đều có mặt. Sẵn sàng xử lý!")
else:
    print("Thiếu file! Cần kiểm tra lại thư mục data/")

KIỂM TRA CÁC FILE CSV CÓ TRONG THƯ MỤC DATA KO
 olist_orders_dataset.csv (16.84 MB)
 olist_order_items_dataset.csv (14.72 MB)
 olist_order_payments_dataset.csv (5.51 MB)
 olist_order_reviews_dataset.csv (13.78 MB)
 olist_customers_dataset.csv (8.62 MB)
 olist_products_dataset.csv (2.27 MB)
 olist_sellers_dataset.csv (0.17 MB)
 olist_geolocation_dataset.csv (58.44 MB)
 product_category_name_translation.csv (0.0 MB)

Tất cả 9/9 file đều có mặt. Sẵn sàng xử lý!


## 1.6 Đọc dữ liệu thô

In [4]:
# Đọc và hiển thị thông tin từng file
for filename in SOURCE_SCHEMA.keys():
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        continue
    
    df = pd.read_csv(filepath, encoding='utf-8-sig', nrows=5000)
    total_rows = sum(1 for _ in open(filepath, encoding='utf-8-sig')) - 1  # Đếm tổng dòng
    
    print("=" * 60)
    print(f" {filename}")
    print(f"   Tổng số dòng: {total_rows:,}")
    print(f"   Số cột: {len(df.columns)}")
    print(f"   Cột: {list(df.columns)}")
    print(f"   Mẫu dữ liệu:")
    display(df.head(3))
    print()

 olist_orders_dataset.csv
   Tổng số dòng: 99,441
   Số cột: 8
   Cột: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
   Mẫu dữ liệu:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



 olist_order_items_dataset.csv
   Tổng số dòng: 112,650
   Số cột: 7
   Cột: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
   Mẫu dữ liệu:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87



 olist_order_payments_dataset.csv
   Tổng số dòng: 103,886
   Số cột: 5
   Cột: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
   Mẫu dữ liệu:


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



 olist_geolocation_dataset.csv
   Tổng số dòng: 1,000,163
   Số cột: 5
   Cột: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
   Mẫu dữ liệu:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP



 product_category_name_translation.csv
   Tổng số dòng: 71
   Số cột: 2
   Cột: ['product_category_name', 'product_category_name_english']
   Mẫu dữ liệu:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 1.7 Kiểm tra chất lượng dữ liệu (Data Quality Check)

### Quy trình kiểm tra:
1. Kiểm tra cột bắt buộc có tồn tại không
2. Đếm số null ở các cột quan trọng
3. Kiểm tra trùng lặp khóa chính

In [5]:
def validate_csv(filepath, schema):
    """Kiểm tra chất lượng một file CSV."""
    filename = os.path.basename(filepath)
    report = {
        "filename": filename,
        "valid": False,
        "row_count": 0,
        "column_count": 0,
        "missing_columns": [],
        "null_counts": {},
        "duplicate_keys": 0,
        "errors": [],
    }
    
    df = pd.read_csv(filepath, encoding='utf-8-sig')
    report["row_count"] = len(df)
    report["column_count"] = len(df.columns)
    
    # 1. Kiểm tra cột bắt buộc
    required = schema["required_columns"]
    missing = [c for c in required if c not in df.columns]
    report["missing_columns"] = missing
    if missing:
        report["errors"].append(f"Thiếu cột: {missing}")
    
    # 2. Đếm null ở các cột bắt buộc
    for col in required:
        if col in df.columns:
            null_count = df[col].isna().sum()
            if null_count > 0:
                pct = round(null_count / len(df) * 100, 1)
                report["null_counts"][col] = f"{null_count:,} ({pct}%)"
    
    # 3. Kiểm tra trùng lặp khóa chính
    pk = schema.get("primary_key")
    if pk and pk in df.columns:
        dupes = df[pk].duplicated().sum()
        report["duplicate_keys"] = dupes
        if dupes > 0:
            report["errors"].append(f"Trùng lặp khóa '{pk}': {dupes:,}")
    
    if not report["errors"]:
        report["valid"] = True
    
    return report

# Chạy validation cho tất cả file
print("=" * 60)
print("KẾT QUẢ KIỂM TRA CHẤT LƯỢNG DỮ LIỆU")
print("=" * 60)

reports = []
for filename, schema in SOURCE_SCHEMA.items():
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        continue
    report = validate_csv(filepath, schema)
    reports.append(report)
    
    status = "PASS" if report["valid"] else "FAIL"
    print(f"\n{status} | {filename}")
    print(f"  Dòng: {report['row_count']:,} | Cột: {report['column_count']}")
    if report["null_counts"]:
        print(f"   Null values: {report['null_counts']}")
    if report["duplicate_keys"] > 0:
        print(f"   Khóa chính trùng: {report['duplicate_keys']:,}")
    if report["missing_columns"]:
        print(f"   Thiếu cột: {report['missing_columns']}")

KẾT QUẢ KIỂM TRA CHẤT LƯỢNG DỮ LIỆU

PASS | olist_orders_dataset.csv
  Dòng: 99,441 | Cột: 8

PASS | olist_order_items_dataset.csv
  Dòng: 112,650 | Cột: 7

PASS | olist_order_payments_dataset.csv
  Dòng: 103,886 | Cột: 5

FAIL | olist_order_reviews_dataset.csv
  Dòng: 99,224 | Cột: 7
   Khóa chính trùng: 814

PASS | olist_customers_dataset.csv
  Dòng: 99,441 | Cột: 5

PASS | olist_products_dataset.csv
  Dòng: 32,951 | Cột: 9
   Null values: {'product_category_name': '610 (1.9%)'}

PASS | olist_sellers_dataset.csv
  Dòng: 3,095 | Cột: 4

PASS | olist_geolocation_dataset.csv
  Dòng: 1,000,163 | Cột: 5

PASS | product_category_name_translation.csv
  Dòng: 71 | Cột: 2


## 1.8 Thống kê tổng quan dữ liệu

In [6]:
# Tạo bảng tổng kết
summary_data = []
for r in reports:
    summary_data.append({
        "File": r["filename"],
        "Số dòng": f"{r['row_count']:,}",
        "Số cột": r["column_count"],
        "Hợp lệ": "✅" if r["valid"] else "❌",
        "Null": "Có" if r["null_counts"] else "Không",
        "Trùng PK": r["duplicate_keys"],
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

total_rows = sum(r["row_count"] for r in reports)
valid_count = sum(1 for r in reports if r["valid"])
print(f"\nTổng số dòng dữ liệu: {total_rows:,}")
print(f"File hợp lệ: {valid_count}/{len(reports)}")

,File,Số dòng,Số cột,Hợp lệ,Null,Trùng PK
0,olist_orders_dataset.csv,"99,441",8,✅,Không,0
1,olist_order_items_dataset.csv,"112,650",7,✅,Không,0
2,olist_order_payments_dataset.csv,"103,886",5,✅,Không,0
3,olist_order_reviews_dataset.csv,"99,224",7,❌,Không,814
4,olist_customers_dataset.csv,"99,441",5,✅,Không,0
5,olist_products_dataset.csv,"32,951",9,✅,Có,0
6,olist_sellers_dataset.csv,"3,095",4,✅,Không,0
7,olist_geolocation_dataset.csv,"1,000,163",5,✅,Không,0
8,product_category_name_translation.csv,71,2,✅,Không,0



Tổng số dòng dữ liệu: 1,550,922
File hợp lệ: 8/9


## 1.9 Upload lên HDFS (Bronze Layer)

Sau khi kiểm tra xong, dữ liệu được upload lên HDFS.

```
HDFS Path: /user/bigdata/olist/bronze/<tên_bảng>/<tên_file.csv>
```

In [7]:
# === MINH HỌA LOGIC UPLOAD LÊN HDFS ===
# Code thực tế nằm trong file: spark_jobs/ingestion.py

# import subprocess
# HADOOP_CMD = "C:/hadoop/bin/hadoop.cmd"
# 
# def hdfs_mkdir(path):
#     """Tạo thư mục trên HDFS."""
#     cmd = [HADOOP_CMD, "dfs", "-mkdir", "-p", path]
#     subprocess.run(cmd, capture_output=True, text=True)
# 
# def hdfs_upload(local_path, hdfs_path):
#     """Upload file từ máy local lên HDFS."""
#     subprocess.run([HADOOP_CMD, "dfs", "-rm", "-f", hdfs_path], capture_output=True)
#     cmd = [HADOOP_CMD, "dfs", "-put", "-f", local_path, hdfs_path]
#     result = subprocess.run(cmd, capture_output=True, text=True)
#     if result.returncode != 0:
#         raise RuntimeError(f"Upload thất bại: {result.stderr}")
#
# # Upload từng file
# for filename, schema in SOURCE_SCHEMA.items():
#     hdfs_dir = f"{HDFS_BRONZE}/{schema['hdfs_folder']}"
#     hdfs_path = f"{hdfs_dir}/{filename}"
#     hdfs_mkdir(hdfs_dir)
#     hdfs_upload(os.path.join(DATA_DIR, filename), hdfs_path)
#     print(f" Uploaded: {filename} -> {hdfs_path}")

print("Đoạn code trên đã được chạy thực tế qua file ingestion.py")
print("Kết quả: 9/9 file đã upload thành công lên HDFS Bronze layer")

Đoạn code trên đã được chạy thực tế qua file ingestion.py
Kết quả: 9/9 file đã upload thành công lên HDFS Bronze layer
